# This is a sample Jupyter Notebook

Below is an example of a code cell. 
Put your cursor into the cell and press Shift+Enter to execute it and select the next one, or click 'Run Cell' button.

Press Double Shift to search everywhere for classes, files, tool windows, actions, and settings.

To learn more about Jupyter Notebooks in PyCharm, see [help](https://www.jetbrains.com/help/pycharm/ipython-notebook-support.html).
For an overview of PyCharm, go to Help -> Learn IDE features or refer to [our documentation](https://www.jetbrains.com/help/pycharm/getting-started.html).

In [87]:
#1 Imports and Warnings Suppression

In [89]:
import requests
from bs4 import BeautifulSoup
from click import prompt
from newspaper import Article
from youtube_transcript_api import YouTubeTranscriptApi
from iText2KG.itext2kg.documents_distiller import DocumentsDistiller, CV
import yaml
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.caches import BaseCache
from pydantic import BaseModel, Field
from langchain_core.callbacks.base import Callbacks
from typing import List
from iText2KG.itext2kg import iText2KG
import networkx as nx
import matplotlib as plt
from langchain.text_splitter import RecursiveCharacterTextSplitter
import time
import warnings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from neo4j_client import Neo4jClient
import traceback
import json
from typing import List

from langchain_core.caches import BaseCache

ChatOllama.BaseCache = BaseCache
ChatOllama.model_rebuild()

# Suppress Pydantic schema warning
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic._internal._generate_schema")

ChatOpenAI.BaseCache = BaseCache
ChatOpenAI.model_rebuild()


In [50]:
#2 Article Scraping Function 

In [51]:
def scrape_article(url):
    print(f"Scraping article from: {url}")

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/114.0.0.0 Safari/537.36"
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()

        # Method 1: BeautifulSoup approach
        soup = BeautifulSoup(response.text, 'html.parser')
        content_div = soup.find("div", class_="entry-content ap-pattern--entry-content")

        if content_div:
            article_text = content_div.get_text(separator="\n", strip=True)
            with open("output/pure_scrum_clean.txt", "w", encoding="utf-8") as f:
                f.write(article_text)
            print("Article scraped successfully with BeautifulSoup")
        else:
            print("Specific div not found, trying newspaper3k...")

        # Method 2: Newspaper3k approach (as backup)
        article = Article(url)
        article.download()
        article.parse()

        if article.text:
            with open("output/pure_scrum_clean2.txt", "w", encoding="utf-8") as f:
                f.write(article.text)
            print("Article scraped successfully with newspaper3k")
            return article.text
        else:
            print("Failed to extract article content")
            return None

    except requests.RequestException as e:
        print(f"Error fetching article: {e}")
        return None

In [52]:
#3. YouTube Transcript Retrieval Function

In [53]:
def get_youtube_transcript(video_id):
    print(f"Getting YouTube transcript for video: {video_id}")

    try:
        transcript_list = YouTubeTranscriptApi.get_transcript(video_id)
        transcript = " ".join([item['text'] for item in transcript_list])

        with open("output/youtubeVideo.txt", "w", encoding="utf-8") as f:
            f.write(transcript)

        print("YouTube transcript downloaded successfully")
        return transcript

    except Exception as e:
        print(f"Error getting YouTube transcript: {e}")
        return None

In [54]:
#4. Configuration Loading

In [55]:
def load_config(config_path="config.yml"):
    try:
        with open(config_path, "r") as f:
            config = yaml.safe_load(f)
        return config
    except FileNotFoundError:
        print("config.yml not found. Creating sample config for Ollama...")

        sample_config = {
            'llm': {
                'model_type': 'ollama',
                'model_name': 'llama3',
                'format': 'json',
                'temperature': 0.7,
                'max_tokens': 2000,
                'timeout': 60,
                'max_retries': 3
            },
            'paths': {
                'output_dir': './output'
            }
        }

        with open(config_path, "w") as f:
            yaml.dump(sample_config, f, default_flow_style=False)

        print("Sample config.yml created. Update it if needed.")
        return sample_config


In [56]:
#5. LLM and Embeddings Initialization

In [57]:
# import os
# 
# 
# def initialize_llm(config):
#     if not config:
#         return None, None
#     
#     llm_config = config.get('llm', {})
#     api_key = llm_config.get('api_key')
#     
#     if not api_key:
#         api_key = os.getenv('OPENAI_API_KEY')
#         if not api_key:
#             print("OpenAI API key not found. Please set OPENAI_API_KEY environment variable or update config.yml")
#             return None, None
#     
#     try:
#         if llm_config.get('model_type') == "openai":
#             llm = ChatOpenAI(
#                 model=llm_config.get('model_name', 'gpt-3.5-turbo'),
#                 temperature=llm_config.get('temperature', 0.7),
#                 max_tokens=llm_config.get('max_tokens', 2000),
#                 api_key=api_key,
#                 request_timeout=llm_config.get('timeout', 60),
#                 max_retries=llm_config.get('max_retries', 3)
#             )
#             embeddings = OpenAIEmbeddings(api_key=api_key)
#             print("LLM and Embeddings initialized successfully")
#             return llm, embeddings
#         else:
#             print("Only 'openai' model type is supported")
#             return None, None
#     except Exception as e:
#         print(f"Error initializing LLM: {e}")
#         return None, None
#     

In [58]:
#5.1 Ollama

In [59]:

def initialize_llm(config):
    if not config:
        return None, None

    llm_config = config.get('llm', {})

    try:
        if llm_config.get('model_type') == "ollama":
            llm = ChatOllama(
                model=llm_config.get('model_name', 'llama3'),
                format=llm_config.get('format', 'json'),
                temperature=llm_config.get('temperature', 0.7)
            )

            embeddings = OllamaEmbeddings(model=llm_config.get('model_name', 'llama3'))

            print("LLM and Embeddings initialized successfully (Ollama)")
            return llm, embeddings
        else:
            print("Only 'ollama' model type is supported in this setup")
            return None, None
    except Exception as e:
        print(f"Error initializing LLM: {e}")
        return None, None


In [60]:
#6. Pydantic Models for Triples

In [61]:
class Triple(BaseModel):
    subject: str
    predicate: str
    object: str


class TriplesOutput(BaseModel):
    triples: List[Triple]


In [62]:
#7. Knowledge Graph Extraction Function with Retry

In [63]:
def simple_kg_extraction(text, llm, limit=8, max_retries=3):
    if not llm:
        print("LLM not initialized")
        return []

    prompt_template = """
Extract key knowledge graph triples from the following text.
Only return triples in the form (subject, predicate, object).
Focus on important entities and relationships. Avoid duplicates.
Output each triple on its own line with this exact format:
- (Entity1, relationship, Entity2)

Text: {input_text}

Return up to {limit} triples.
"""

    text_to_use = text[:1500]

    for attempt in range(max_retries):
        prompt = prompt_template.format(input_text=text_to_use, limit=limit)

        try:
            response = llm.invoke(prompt)
            lines = response.content.split('\n')
            triples = []

            for line in lines:
                line = line.strip()
                if line.startswith('-') and '(' in line and ')' in line:
                    start = line.find('(')
                    end = line.find(')')
                    if start != -1 and end != -1:
                        triple_str = line[start + 1:end]
                        parts = [part.strip() for part in triple_str.split(',')]
                        if len(parts) == 3:
                            triples.append((parts[0], parts[1], parts[2]))

            return triples

        except Exception as e:
            error_msg = str(e).lower()

            if "rate limit" in error_msg or "429" in error_msg:
                wait_time = (2 ** attempt) * 10
                print(f"Rate limit hit (attempt {attempt + 1}/{max_retries}). Waiting {wait_time}s...")
                time.sleep(wait_time)

            elif "token" in error_msg:
                print(f"Token limit exceeded. Reducing text size and retrying...")
                text_to_use = text_to_use[:len(text_to_use) // 2]
                limit = max(3, limit // 2)
                time.sleep(5)

            else:
                print(f"Error in KG extraction (attempt {attempt + 1}/{max_retries}): {e}")
                time.sleep(5)

    print(f"Failed to extract triples after {max_retries} attempts")
    return []


In [64]:
#8. Process Text in Chunks to Avoid Rate Limits

In [65]:
def process_text_in_chunks(text, llm, chunk_size=1000, chunk_overlap=50, delay_between_chunks=0):
    print(f"Processing text in chunks...")

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )
    chunks = splitter.split_text(text)

    print(f"Created {len(chunks)} chunks")

    all_triples = []

    for i, chunk in enumerate(chunks, 1):
        limit = max(5, len(chunk) // 150)
        print(f"\nProcessing chunk {i}/{len(chunks)} with triple limit {limit}")

        triples = simple_kg_extraction(chunk, llm, limit=limit)
        if triples:
            all_triples.extend(triples)
            print(f"Chunk {i} → {len(triples)} triples extracted")
        else:
            print(f"Chunk {i} → No triples extracted")

        if i < len(chunks) and delay_between_chunks > 0:
            print(f"Waiting {delay_between_chunks}s before next chunk...")
            time.sleep(delay_between_chunks)

    return all_triples


In [66]:
#8. Save Triples to File and also filter duplicates

In [96]:
# def normalize_triples(triples):
#     return [(s.strip().lower(), p.strip().lower(), o.strip().lower()) for s, p, o in triples]
# 
# 
# def filter_triples(triples):
#     filtered = []
#     for subj, pred, obj in triples:
#         if subj and pred and obj and subj.lower() != 'none' and obj.lower() != 'none':
#             filtered.append((subj, pred, obj))
#     return filtered
# 
# 
# def remove_duplicates(triples):
#     seen = set()
#     unique_triples = []
#     for triple in triples:
#         if triple not in seen:
#             unique_triples.append(triple)
#             seen.add(triple)
#     return unique_triples
# 
# 
def save_triples(triples, filename="extracted_triples.txt"):
    with open(filename, "w", encoding="utf-8") as f:
        f.write("Extracted Knowledge Graph Triples:\n")
        f.write("=" * 40 + "\n\n")
        for i, (subject, predicate, obj) in enumerate(triples, 1):
            f.write(f"{i}. ({subject}) --[{predicate}]--> ({obj})\n")

    print(f"Triples saved to {filename}")

def normalize_triples(triples):
    normalized = []
    for triple in triples:
        if len(triple) == 3:
            subject, predicate, obj = triple
            subject = str(subject).strip().lower()
            predicate = str(predicate).strip().lower().replace('_', ' ')
            obj = str(obj).strip().lower()
            normalized.append((subject, predicate, obj))
    return normalized


def filter_triples(triples):
    filtered = []
    for triple in triples:
        if len(triple) == 3:
            subject, predicate, obj = triple
            if len(subject) > 1 and len(predicate) > 1 and len(obj) > 1:
                if subject != obj:
                    filtered.append(triple)
    return filtered


def remove_duplicates(triples):
    return list(set(triples))

def semantic_filter(triples):
    hallucinated_terms = {"family", "romantic", "divorce"}
    return [
        t for t in triples
        if not any(term in str(t).lower() for term in hallucinated_terms)
    ]

# def save_triples(triples, filename="extracted_triples.txt"):
#     with open(filename, 'w', encoding='utf-8') as f:
#         for subject, predicate, obj in triples:
#             f.write(f"{subject}\t{predicate}\t{obj}\n")


In [97]:
#11. Plots

In [98]:
import networkx as nx
import matplotlib.pyplot as plt


def plot_triples(triples):
    G = nx.DiGraph()
    for subj, pred, obj in triples:
        G.add_node(subj)
        G.add_node(obj)
        G.add_edge(subj, obj, label=pred)

    pos = nx.spring_layout(G, k=0.5, iterations=50)

    plt.figure(figsize=(12, 8))
    nx.draw(G, pos, with_labels=True, node_color='lightblue', node_size=3000, font_size=10, arrowsize=20)

    edge_labels = {(subj, obj): pred for subj, pred, obj in triples}
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_color='red', font_size=9)

    plt.title("Knowledge Graph Visualization")
    plt.show()


In [99]:
#extract triples

In [100]:
def extract_triples_from_kg_result(kg_result):
    triples = []

    if hasattr(kg_result, 'relationships') and kg_result.relationships:
        print(f"Found {len(kg_result.relationships)} relationships in KG result")

        for rel in kg_result.relationships:
            try:
                if all(hasattr(rel, attr) for attr in ['name', 'startEntity', 'endEntity']):
                    subject = getattr(rel.startEntity, 'name', None)
                    predicate = rel.name
                    obj = getattr(rel.endEntity, 'name', None)

                    if subject and predicate and obj:
                        subject = str(subject).strip().lower()
                        predicate = str(predicate).strip().replace('_', ' ').lower()
                        obj = str(obj).strip().lower()
                        triples.append((subject, predicate, obj))
                    else:
                        print(f"Skipping incomplete relationship: {rel}")

            except Exception as e:
                print(f"Error processing relationship {rel}: {e}")
                continue

    elif hasattr(kg_result, 'entities') and kg_result.entities:
        print(f"No relationships found, but found {len(kg_result.entities)} entities")
        for entity in kg_result.entities:
            if hasattr(entity, 'name') and hasattr(entity, 'label'):
                name = str(entity.name).strip().lower()
                label = str(entity.label).strip().replace('_', ' ').lower()
                triples.append((name, "is_a", label))

    return triples


In [101]:
#9. Main Execution Logic

In [102]:
# import re
# def main():
#     print("Starting Knowledge Graph Extraction Pipeline\n")
# 
#     # Scrape article
#     # url = "https://age-of-product.com/pure-scrum/"
#     # article_text = scrape_article(url)
#     # 
#     # # Get YouTube transcript
#     # video_id = "502ILHjX9EE"
#     # youtube_text = get_youtube_transcript(video_id)
# 
#     # Choose which text to process
#     # if youtube_text:
#     #     input_text = youtube_text
#     #     print("Using YouTube transcript for KG extraction")
#     # elif article_text:
#     #     input_text = article_text
#     #     print("Using article text for KG extraction")
#     # else:
#     #     print("No text available for processing")
#     #     return
# 
#     try:
#         with open("text.txt", "r", encoding="utf-8") as f:
#             input_text = f.read()
#     except Exception as e:
#         print(f"Error reading 'text.txt': {e}")
#         return
# 
#     config = load_config()
#     llm, embeddings = initialize_llm(config)
# 
#     if not llm:
#         print("Cannot proceed without LLM initialization")
#         return
# 
#     print(f"\nInput text length: {len(input_text)} characters")
#     # all_triples = process_text_in_chunks(input_text, llm,delay_between_chunks=0)
# 
#     # shall try this next automation
#     kg = iText2KG(llm,embeddings)
# 
#     doc = {
#         "text": input_text,
#         "title": "ManualText",
#         "meta": {
#             "source": "manual_input"
#         }
#     }
#     try:
#         response = llm.invoke(f"Extract triples from text:\n\n{input_text[:1000]}")
#         print("Raw LLM response:\n", response.content)
# 
#     except Exception as e:
#         print(f"Extraction failed: {e}")
#         return
# 
#     def parse_llm_response_to_triples(text):
#         triples = []
#         pattern = re.compile(r"^\s*\d+\.\s*(.+?)\s*-\s*(.+)$")
#         for line in text.splitlines():
#             match = pattern.match(line)
#             if match:
#                 subj = match.group(1).strip()
#                 rest = match.group(2).strip()
#                 parts = rest.split(' - ')
#                 if len(parts) == 2:
#                     pred = parts[0].strip()
#                     obj = parts[1].strip()
#                 else:
#                     pred = rest
#                     obj = ""
#                 triples.append((subj, pred, obj))
#         return triples
# 
#     
#     all_triples = parse_llm_response_to_triples(response.content)
# 
#     if all_triples:
#         print(f"\nTotal triples extracted: {len(all_triples)}")
#         all_triples = normalize_triples(all_triples)
#         all_triples = filter_triples(all_triples)
#         all_triples = remove_duplicates(all_triples)
#         save_triples(all_triples)
# 
#         print(Neo4jClient.__init__)
#         neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
#         neo4j_client.create_triples(all_triples)
#         neo4j_client.close()
# 
#         print("\nSample triples:")
#         for i, (subject, predicate, obj) in enumerate(all_triples[:5], 1):
#             print(f"  {i}. ({subject}) --[{predicate}]--> ({obj})")
# 
#         if len(all_triples) > 5:
#             print(f"  ... and {len(all_triples) - 5} more triples")
# 
#         plot_triples(all_triples)
#     else:
#         print("No triples were successfully extracted")




In [127]:
import traceback

def main():
    print("Starting Knowledge Graph Extraction Pipeline\n")

    try:
        with open("text.txt", "r", encoding="utf-8") as f:
            input_text = f.read()
        print(f"Loaded text: {len(input_text)} characters")
        print(f"Text content: {input_text}")  

        if len(input_text) < 100:
            print("WARNING: Text is very short. Consider using longer content for better extraction.")

    except Exception as e:
        print(f"Error reading 'text.txt': {e}")
        return

    config = load_config()
    llm, embeddings = initialize_llm(config)

    if not llm:
        print("Cannot proceed without LLM initialization")
        return

    print(f"LLM initialized: {config['llm']['model_name']}")

    # CRITICAL: Check if your LLM has JSON formatting capability
    print("\n=== TESTING LLM JSON CAPABILITY ===")
    try:
        test_prompt = f"""
        Extract information from this text and return ONLY a JSON object:
        Text: {input_text}
        
        Return format:
        {{
            "entities": ["entity1", "entity2"],
            "summary": "brief summary"
        }}
        """

        if hasattr(llm, 'invoke'):
            test_response = llm.invoke(test_prompt)
            print("\n=== RAW LLM OUTPUT ===")
            print(test_response.content)
            print(f"LLM test response: {test_response.content[:200]}...")

            json_capable = False
            try:
                json.loads(test_response.content.strip())
                print("LLM produces pure JSON")
                json_capable = True
            except json.JSONDecodeError:
                try:
                    import re
                    json_match = re.search(r'\{.*\}', test_response.content, re.DOTALL)
                    if json_match:
                        json_content = json_match.group(0)
                        json.loads(json_content)
                        print("LLM produces JSON wrapped in text (this is normal for Ollama)")
                        json_capable = True
                    else:
                        print("No JSON found in LLM response")
                        json_capable = False
                except json.JSONDecodeError:
                    print("LLM does not produce valid JSON format")
                    print("This is likely why DocumentsDistiller is failing")
                    json_capable = False
        else:
            json_capable = False

    except Exception as e:
        print(f"LLM JSON test failed: {e}")
        json_capable = False

    if not json_capable:
        print("\n=== OLLAMA CONFIGURATION ISSUE ===")
        print(f"Your Ollama {config['llm']['model_name']} model is producing JSON wrapped in text.")
        print("This is normal for Ollama, but DocumentsDistiller expects pure JSON.")
       

    doc = {
        "text": input_text,
        "title": "ManualText",
        "meta": {
            "source": "manual_input",
            "domain": "general",
            "type": "text"
        }
    }

    all_triples = []
    distilled_docs = []  

    print("\n=== PHASE 1: DOCUMENT DISTILLATION ===")

    try:
        print("Initializing Document Distiller...")
        distiller = DocumentsDistiller(llm)

        class OutputStructure(BaseModel):
            entities: List[str] = Field(default_factory=list)
            relationships: List[str] = Field(default_factory=list)
            triples: List[str] = Field(default_factory=list)
            summary: str = Field(default="")

        IE_query = """
        You are an information extraction engine. Carefully read the input and extract a structured summary grounded only in the text. Output STRICTLY a JSON object with:

        {
          "entities": [list of concrete people, places, concepts from the text],
          "relationships": [list of meaningful relations between those entities],
          "triples": ["Subject-Predicate-Object"],
          "summary": "one-sentence summary of the input"
        }

        IMPORTANT:
        - DO NOT infer or invent unrelated knowledge.
        - Base ALL output strictly on the provided input text.
        - Return only valid JSON. Do not include explanations or text outside the JSON object.
        """

        print("Starting distillation process with simplified approach...")

        raw_output = distiller.distill([doc["text"]], OutputStructure, IE_query)

        print("\n=== RAW OUTPUT FROM distiller.distill() ===")
        print(raw_output)

        print("LLM RAW RESPONSE:")
        print(llm.invoke(IE_query).content)

        if raw_output and len(raw_output) > 0:
            if isinstance(raw_output, dict):
                raw_output = [raw_output]
    
            print("Raw output items:")
            for i, item in enumerate(raw_output):
                print(f"  Item {i}: {type(item)} -> {item}")

            valid_outputs = []
            for item in raw_output:
                if item is not None and isinstance(item, dict):
                    has_content = False
                    for key in ['entities', 'relationships', 'triples', 'summary']:
                        value = item.get(key, None)
                        if (isinstance(value, list) and len(value) > 0) or (isinstance(value, str) and len(value) > 5):
                            has_content = True
                            break
                    if has_content:
                        valid_outputs.append(item)

            if valid_outputs:
                print(f"\nFound {len(valid_outputs)} valid distilled documents:")
                for i, out in enumerate(valid_outputs):
                    print(f"\n--- Distilled Document #{i + 1} ---")
                    print(json.dumps(out, indent=2))
                distilled_docs = valid_outputs
            else:
                print("\nNo valid content in distilled outputs")
                print("DocumentsDistiller is not working properly with your setup")
                return []
        else:
            print("\nDistiller returned empty or None result")
            return []

    except Exception as e:
        print(f"\nDocument distillation failed: {e}")
        traceback.print_exc()

        if "'NoneType' object has no attribute 'items'" in str(e):
            print("\n=== SPECIFIC ISSUE IDENTIFIED ===")
            print("The DocumentsDistiller is receiving None from your LLM instead of JSON.")

        return []

    if not distilled_docs:
        print("No distilled documents to process for KG construction")
        return []

    print("\n=== PHASE 2: KNOWLEDGE GRAPH CONSTRUCTION ===")

    try:
        print("Initializing iText2KG...")
        kg_extractor = iText2KG(llm, embeddings)

        print("Building knowledge graph...")
        texts_for_kg = []

        for i, doc_item in enumerate(distilled_docs):
            print(f"Processing distilled doc {i + 1}: {type(doc_item)}")

            if isinstance(doc_item, dict):
                combined_text = ""

                if doc_item.get('summary'):
                    combined_text += f"Summary: {doc_item['summary']}. "

                if doc_item.get('entities'):
                    combined_text += f"Entities: {', '.join(doc_item['entities'])}. "

                if doc_item.get('relationships'):
                    combined_text += f"Relationships: {'. '.join(doc_item['relationships'])}. "

                if doc_item.get('triples'):
                    triple_strings = [' '.join(triple) if isinstance(triple, list) else str(triple) for triple in doc_item['triples']]
                    combined_text += f"Triples: {'. '.join(triple_strings)}."

                if combined_text:
                    texts_for_kg.append(combined_text)
                elif 'text' in doc_item:
                    texts_for_kg.append(doc_item['text'])

            elif isinstance(doc_item, str) and len(doc_item.strip()) > 10:
                texts_for_kg.append(doc_item)

        print(f"Prepared {len(texts_for_kg)} texts for KG construction")

        if not texts_for_kg:
            print("ERROR: No valid text prepared for KG construction")
            return []

        for i, text in enumerate(texts_for_kg):
            print(f"KG Input {i + 1}: {text[:150]}...")

        kg_result = kg_extractor.build_graph(texts_for_kg)

        print(f"=== DEBUGGING KG RESULT ===")
        print(f"KG result type: {type(kg_result)}")

        if hasattr(kg_result, '__dict__'):
            print(f"KG result attributes: {list(kg_result.__dict__.keys())}")

        all_triples = extract_triples_from_kg_result(kg_result)
        print(f"Successfully extracted {len(all_triples)} triples")

    except Exception as e:
        print(f"Knowledge graph construction failed: {e}")
        traceback.print_exc()
        return []

    print("\n=== PHASE 3: POST-PROCESSING ===")

    if all_triples:
        original_count = len(all_triples)

        all_triples = normalize_triples(all_triples)
        all_triples = filter_triples(all_triples)
        all_triples = remove_duplicates(all_triples)
        all_triples = semantic_filter(all_triples)
        print(f"After cleaning: {len(all_triples)} triples (removed {original_count - len(all_triples)})")

        save_triples(all_triples, "extracted_triples.txt")
        print("Triples saved to file")

        try:
            print("Saving to Neo4j...")
            neo4j_client = Neo4jClient(uri="bolt://localhost:7687", user="neo4j", password="12345678")
            neo4j_client.create_triples(all_triples)
            neo4j_client.close()
            print("Successfully saved to Neo4j")
        except Exception as e:
            print(f"Neo4j storage failed: {e}")

        print("\n=== EXTRACTED KNOWLEDGE GRAPH ===")
        for i, (subject, predicate, obj) in enumerate(all_triples[:10], 1):
            print(f"  {i:2d}. ({subject}) --[{predicate}]--> ({obj})")
        if len(all_triples) > 10:
            print(f"  ... and {len(all_triples) - 10} more triples")

        try:
            plot_triples(all_triples)
            print("Visualization created")
        except Exception as e:
            print(f"Visualization failed: {e}")

        return all_triples

    else:
        print("\nNo triples were successfully extracted")
        return []


In [128]:
#10. Run the main() function

In [129]:
if __name__ == "__main__":
    main()


Starting Knowledge Graph Extraction Pipeline

Loaded text: 978 characters
Text content: Let's talk about Agile Software development from the perspective of the Product Owner Here's Pat, she is a Product Owner. She has a product vision that she is really passionate about She doesn't know the details of what our product is going to do, but she knows why we are building the product What problem it's gonna solve, and for who She talks about all the time. Here are the stakeholders They are the people who are going to use and support or in any way be affected by the system being developed. Pat's vision is that these people here will love our system and use it all the time and tell their friends about it. The stakeholders needs and Pat's ideas are expressed as user stories, here. For example if this is was flight booking system people need to be able to search for a
flight, and maybe that would be one user story Both Pat and the stakeholders
have lots of ideas so Pat helps turn these into con